# 06 — ToN-IoT Ensemble Development

**Goal:** Check if combining LightGBM, Decision Tree and Random Forest improves detection.

**Input:** Saved Core10 models + frozen validation data. **No retraining. No test access.**

**Output:** Error diversity, ensemble scores, attack-type analysis, resource costs and two graphs.

## 1. Verify inputs

Use the saved split and the same 10 model features.

In [ ]:
from pathlib import Path
from time import perf_counter
from datetime import datetime, timezone
from threading import Thread, Event
from itertools import combinations
import os, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, psutil
from IPython.display import display
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

ROOT = Path(r'D:\J26-CS-336-Network-Flow-V2\network_flow_v2')
VALID = ROOT / 'data/splits/ton_iot/ton_iot_core10_validation_v3.csv.gz'
MANIFEST = ROOT / 'data/splits/ton_iot/ton_iot_core10_signature_split_manifest_v3.csv'
SPLIT_REPORT = ROOT / 'reports/audits/ton_iot_v3_split_report.json'
MODEL_DIR = ROOT / 'models/ton_iot_v3'
AUDITS = ROOT / 'reports/audits'
METRICS = ROOT / 'reports/metrics'
FIGURES = ROOT / 'reports/figures'
SHORTLIST = ('lightgbm', 'decision_tree', 'random_forest')
EXPECTED = {'train':147730, 'validation':31657, 'test':31656}
MISSING = {'', '-', '?', 'na', 'n/a', 'nan', 'none', 'null', 'missing', 'unknown', '<missing>'}

for p in (VALID, MANIFEST, SPLIT_REPORT):
    if not p.is_file():
        raise FileNotFoundError(f'Missing previous dataset file: {p}')
split = json.loads(SPLIT_REPORT.read_text(encoding='utf-8'))
if split.get('dataset') != 'ToN-IoT' or split.get('test_scored') is True:
    raise ValueError('Invalid ToN-IoT split report or already-scored test.')
if split.get('normalized_fingerprint_overlap_groups') != 0:
    raise ValueError('Overlapping Core10 groups: stop.')
counts = {key:int(split['counts'][key]['rows']) for key in EXPECTED}
if counts != EXPECTED or split.get('total_rows_preserved') != 211043:
    raise ValueError(f'Frozen counts have changed: {counts}')
sha = hashlib.sha256()
with MANIFEST.open('rb') as stream:
    for block in iter(lambda:stream.read(8*1024*1024), b''):
        sha.update(block)
if sha.hexdigest() != split['manifest_sha256']:
    raise ValueError('Frozen split manifest changed.')
FEATURES = list(split['model_features'])
CAT = list(split['categorical_features'])
NUM = [c for c in FEATURES if c not in CAT]
if len(FEATURES) != 10 or len(CAT) != 3 or len(NUM) != 7:
    raise ValueError('Unexpected ToN-IoT Core10 features.')
for d in (AUDITS, METRICS, FIGURES):
    d.mkdir(parents=True, exist_ok=True)
print('ToN-IoT Core10 | Validation:',EXPECTED['validation'],'| TEST NOT READ')

## 2. Load validation records

Keep the full 31,657 records. Labels are for scoring, not model inputs.

In [ ]:
frame = pd.read_csv(VALID, dtype=str, keep_default_na=False, low_memory=False)
fields = ['csv_row_index_0based'] + FEATURES + ['label', 'type']
if frame.columns.tolist() != fields or len(frame) != EXPECTED['validation']:
    raise ValueError('Validation columns or row count changed.')
if frame['csv_row_index_0based'].duplicated().any():
    raise ValueError('Repeated original row identifiers.')
if not frame['label'].isin(['0','1']).all():
    raise ValueError('Unknown Normal / Attack labels.')
y = frame['label'].astype(np.uint8).to_numpy()
attack, normal = y == 1, y == 0
if int(attack.sum()) != 24157 or int(normal.sum()) != 7500:
    raise ValueError('Unexpected class counts.')
types = frame['type'].astype(str).str.strip().str.casefold().to_numpy()
if np.any(normal & (types != 'normal')) or np.any(attack & (types == 'normal')):
    raise ValueError('Attack type disagrees with binary label.')
service_missing = frame['service'].astype(str).str.strip().str.casefold().isin(MISSING).to_numpy()
protocol = frame['proto'].astype(str).str.strip().str.casefold().to_numpy()

def clean_features(raw):
    x = raw[FEATURES].copy()
    for col in NUM:
        txt = x[col].astype(str).str.strip()
        txt = txt.mask(txt.str.casefold().isin(MISSING))
        x[col] = pd.to_numeric(txt, errors='coerce').replace([np.inf,-np.inf],np.nan).astype(np.float32)
    for col in CAT:
        txt = x[col].astype(str).str.strip()
        x[col] = txt.mask(txt.str.casefold().isin(MISSING), '__MISSING__')
    return x

X = clean_features(frame)
print(f'Validation: {len(y):,} | Normal: {normal.sum():,} | Attack: {attack.sum():,}')

## 3. Verify the three saved classifiers

Compare their predictions with Notebook 04 results before combining them.

In [ ]:
models, predictions, probabilities = {}, {}, {}
for name in SHORTLIST:
    path = MODEL_DIR / f'{name}_core10_v3.joblib'
    audit_path = AUDITS / f'ton_iot_v3_{name}_training.json'
    for p in (path,audit_path):
        if not p.is_file():
            raise FileNotFoundError(f'Run Notebook 04 first: {p}')
    model = joblib.load(path)  # Only load models you saved yourself.
    if not hasattr(model, 'named_steps') or not {'preprocessing','classifier'}.issubset(model.named_steps):
        raise ValueError(f'{name}: model is not the saved complete preprocessing pipeline.')
    if list(model.classes_) != [0,1]:
        raise ValueError(f'{name}: unexpected class order.')
    p = np.asarray(model.predict(X),dtype=np.uint8)
    proba = np.asarray(model.predict_proba(X)[:,1],dtype=float)
    if len(p) != len(y) or not np.isin(p,[0,1]).all() or not np.isfinite(proba).all():
        raise ValueError(f'{name}: invalid model predictions.')
    old = json.loads(audit_path.read_text(encoding='utf-8'))
    if (old.get('test_scored') is not False or
        old.get('frozen_manifest_sha256') != split['manifest_sha256'] or
        old.get('features') != FEATURES):
        raise ValueError(f'{name}: model report does not match this frozen experiment.')
    cm = tuple(int(v) for v in confusion_matrix(y,p,labels=[0,1]).ravel())
    expected_cm = tuple(int(old['metrics'][k]) for k in ('tn','fp','fn','tp'))
    if cm != expected_cm:
        raise ValueError(f'{name}: validation confusion matrix changed: {cm} != {expected_cm}')
    models[name], predictions[name], probabilities[name] = model,p,proba
    print(f'{name:14s} FP={cm[1]:,} FN={cm[2]:,} F1={100*f1_score(y,p):.3f}%')
print('All three models match Notebook 04.')

## 4. Do the models make different mistakes?

A **rescued attack** is missed by one model but correctly detected by another.

In [ ]:
rows=[]
for a,b in combinations(SHORTLIST,2):
    pa,pb=predictions[a],predictions[b]
    miss_a=attack & (pa==0)
    miss_b=attack & (pb==0)
    fp_a=normal & (pa==1)
    fp_b=normal & (pb==1)
    rows.append({
        'first_model':a, 'second_model':b,
        'both_miss_same_attacks':int((miss_a & miss_b).sum()),
        'first_missed_rescued_by_second':int((miss_a & (pb==1)).sum()),
        'second_missed_rescued_by_first':int((miss_b & (pa==1)).sum()),
        'new_false_alarms_if_OR':int((~fp_a & fp_b).sum()),
        'shared_false_alarms':int((fp_a & fp_b).sum())
    })
diversity=pd.DataFrame(rows)
display(diversity)

## 5. Compare individual models and ensembles

OR catches an attack if **any** model says Attack. Majority needs **two of three** models.

In [ ]:
lgb,dt,rf=(predictions[name] for name in SHORTLIST)
variants={
    'LightGBM':lgb,
    'Decision Tree':dt,
    'Random Forest':rf,
    'LightGBM + RF (OR)':(lgb|rf).astype(np.uint8),
    'LightGBM + DT (OR)':(lgb|dt).astype(np.uint8),
    'LightGBM + RF (AND)':(lgb&rf).astype(np.uint8),
    'Three-model majority':((lgb.astype(int)+dt.astype(int)+rf.astype(int))>=2).astype(np.uint8),
    'Three-model OR':(lgb|dt|rf).astype(np.uint8),
    'Three-model mean probability':((probabilities['lightgbm']+probabilities['decision_tree']+
                                    probabilities['random_forest'])/3 >= 0.5).astype(np.uint8)
}

def metrics_for(label,p):
    tn,fp,fn,tp=(int(v) for v in confusion_matrix(y,p,labels=[0,1]).ravel())
    return {'candidate':label,'attack_f1_pct':round(100*f1_score(y,p,zero_division=0),3),
        'attack_precision_pct':round(100*precision_score(y,p,zero_division=0),3),
        'attack_recall_pct':round(100*recall_score(y,p,zero_division=0),3),
        'false_positive_pct':round(100*fp/max(tn+fp,1),3),
        'accuracy_pct':round(100*accuracy_score(y,p),3),
        'false_alarms':fp,'missed_attacks':fn,'tn':tn,'fp':fp,'fn':fn,'tp':tp}

comparison=pd.DataFrame([metrics_for(k,p) for k,p in variants.items()])
display(comparison[['candidate','attack_f1_pct','false_positive_pct',
                    'attack_recall_pct','false_alarms','missed_attacks']].sort_values('attack_f1_pct',ascending=False))
print('Higher F1 does not automatically mean lower resource cost or better MITM detection.')

## 6. Attack types and missing service

Check MITM, other attacks, and missing-service cases separately.

In [ ]:
attack_types=[]
for label,p in variants.items():
    for kind in sorted(set(types[attack])):
        mask=attack & (types==kind)
        total=int(mask.sum()); detected=int((p[mask]==1).sum())
        attack_types.append({'candidate':label,'attack_type':kind,
                             'attack_records':total,'detected':detected,'missed':total-detected,
                             'recall_pct':round(100*detected/total,2)})
by_type=pd.DataFrame(attack_types)
display(by_type[by_type['attack_type'].isin(['mitm','ransomware','ddos'])][
    ['candidate','attack_type','attack_records','detected','missed','recall_pct']
].sort_values(['attack_type','candidate']))

service_rows=[]
for label,p in variants.items():
    for status,flag in [('missing',True),('present',False)]:
        for cls,labelval in [('Normal',0),('Attack',1)]:
            mask=(service_missing==flag) & (y==labelval)
            num=int(mask.sum()); errors=int((p[mask]!=labelval).sum())
            service_rows.append({'candidate':label,'service':status,'actual_class':cls,
               'records':num,'errors':errors,'error_pct':round(100*errors/num,3) if num else np.nan,
               'metric':'FPR' if labelval==0 else 'FNR'})
service_result=pd.DataFrame(service_rows)
display(service_result[service_result['candidate'].isin([
    'LightGBM','LightGBM + RF (OR)','Three-model majority'])])

## 7. Check training vs validation `service` availability

Different missing-value patterns can affect evaluation. This reads training and validation only.

In [ ]:
TRAIN_CSV = ROOT / 'data/splits/ton_iot/ton_iot_core10_train_v3.csv.gz'
if not TRAIN_CSV.is_file():
    raise FileNotFoundError(f'Missing frozen training partition: {TRAIN_CSV}')
train_service = pd.read_csv(TRAIN_CSV, usecols=['label','service'],
                            dtype=str, keep_default_na=False, low_memory=False)
if len(train_service) != EXPECTED['train'] or not train_service['label'].isin(['0','1']).all():
    raise ValueError('Training partition counts or labels changed.')
shift_rows=[]
for partition, data in [('train',train_service),('validation',frame[['label','service']])]:
    missing=data['service'].astype(str).str.strip().str.casefold().isin(MISSING)
    for class_value,class_name in [('0','Normal'),('1','Attack')]:
        mask=data['label'].eq(class_value)
        total=int(mask.sum()); missing_n=int((missing & mask).sum())
        shift_rows.append({'partition':partition,'class':class_name,
            'records':total,'service_missing_count':missing_n,
            'service_missing_pct':round(100*missing_n/total,2) if total else np.nan})
shift=pd.DataFrame(shift_rows)
display(shift)
for cls in ['Normal','Attack']:
    t=float(shift.loc[(shift['partition']=='train') & (shift['class']==cls),'service_missing_pct'].iloc[0])
    v=float(shift.loc[(shift['partition']=='validation') & (shift['class']==cls),'service_missing_pct'].iloc[0])
    if abs(v-t) >= 10:
        print(f'CAUTION: {cls} service-missing rate differs by {v-t:+.2f} percentage points across train/validation.')
print('Train/validation diagnostic only; held-out test remains sealed.')

## 8. Protocol errors

These are small-group diagnostics; treat ICMP results cautiously.

In [ ]:
protocol_rows=[]
for label,p in variants.items():
    for proto in sorted(set(protocol)):
        mask=protocol==proto
        num=int(mask.sum()); err=int((p[mask]!=y[mask]).sum())
        protocol_rows.append({'candidate':label,'protocol':proto,'records':num,'errors':err,
            'error_pct':round(100*err/num,2) if num else np.nan})
protocol_result=pd.DataFrame(protocol_rows)
display(protocol_result[protocol_result['candidate'].isin([
    'LightGBM','LightGBM + RF (OR)','Three-model majority'])].sort_values(['protocol','candidate']))

## 9. Measure ensemble resource cost

Benchmark the full pipeline on this laptop. Predictions include preprocessing. RAM includes the whole Python process.

In [ ]:
from time import perf_counter
from threading import Event, Thread

proc=psutil.Process(os.getpid())
BENCH_RUNS=60
specs={
  'LightGBM':('lightgbm',), 'Decision Tree':('decision_tree',),
  'Random Forest':('random_forest',),
  'LightGBM + RF (OR)':('lightgbm','random_forest'),
  'LightGBM + DT (OR)':('lightgbm','decision_tree'),
  'LightGBM + RF (AND)':('lightgbm','random_forest'),
  'Three-model majority':SHORTLIST,
  'Three-model OR':SHORTLIST,
  'Three-model mean probability':SHORTLIST
}

def predict_ensemble(name,x):
    member_names=specs[name]
    if name=='Three-model mean probability':
        p=sum(models[m].predict_proba(x)[:,1] for m in member_names)/len(member_names)
        return (p>=0.5).astype(np.uint8)
    votes=[np.asarray(models[m].predict(x),dtype=np.uint8) for m in member_names]
    if len(votes)==1: return votes[0]
    if name.endswith('(OR)') or name=='Three-model OR':
        return np.logical_or.reduce(votes).astype(np.uint8)
    if name.endswith('(AND)'):
        return np.logical_and.reduce(votes).astype(np.uint8)
    if name=='Three-model majority':
        return (sum(v.astype(np.int32) for v in votes)>=2).astype(np.uint8)
    raise ValueError(name)

def cpu_time():
    t=proc.cpu_times()
    return t.user+t.system

def batch_benchmark(label):
    peak=[proc.memory_info().rss]
    stop=Event()
    def sampler():
        while not stop.wait(0.025):
            peak[0]=max(peak[0],proc.memory_info().rss)
    thread=Thread(target=sampler,daemon=True)
    c0=cpu_time();start=perf_counter()
    thread.start()
    try:
        predicted=predict_ensemble(label,X)
    finally:
        elapsed=perf_counter()-start
        cpu=max(0.0,cpu_time()-c0)
        stop.set();thread.join()
        peak[0]=max(peak[0],proc.memory_info().rss)
    if not np.array_equal(predicted,variants[label]):
        raise ValueError(f'{label}: benchmark output differs from validation results')
    return elapsed,100*cpu/max(elapsed,1e-9),peak[0]/(1024**2)

resource_rows=[]
indices=np.linspace(0,len(X)-1,num=min(BENCH_RUNS,len(X)),dtype=int)
for label in variants:
    for _ in range(3):predict_ensemble(label,X.iloc[[int(indices[0])]])
    samples=[]
    for idx in indices:
        single=X.iloc[[int(idx)]]
        t0=perf_counter()
        predict_ensemble(label,single)
        samples.append((perf_counter()-t0)*1000)
    sec,cpu,memory=batch_benchmark(label)
    storage=sum((MODEL_DIR/f'{m}_core10_v3.joblib').stat().st_size for m in specs[label])/(1024**2)
    resource_rows.append({'candidate':label,'model_count':len(specs[label]),
        'single_flow_p50_ms':round(float(np.percentile(samples,50)),3),
        'single_flow_p95_ms':round(float(np.percentile(samples,95)),3),
        'single_flow_runs':len(samples),'validation_batch_seconds':round(sec,4),
        'batch_throughput_flows_per_second':round(len(y)/max(sec,1e-9),1),
        'process_cpu_pct_during_batch':round(cpu,2),
        'peak_process_ram_mib':round(memory,2),
        'combined_model_storage_mib':round(storage,4)})
resources=pd.DataFrame(resource_rows)
display(resources[['candidate','model_count','single_flow_p95_ms',
                   'batch_throughput_flows_per_second','combined_model_storage_mib','peak_process_ram_mib']])
print('Laptop process measurements, not Raspberry Pi or end-to-end packet processing.')

## 10. Save results

Two graphs, seven CSV tables and one JSON summary. No test predictions.

In [ ]:
shift.to_csv(METRICS/'ton_iot_v3_service_train_validation_shift.csv',index=False)
comparison.to_csv(METRICS/'ton_iot_v3_ensemble_comparison.csv',index=False)
diversity.to_csv(METRICS/'ton_iot_v3_model_error_diversity.csv',index=False)
by_type.to_csv(METRICS/'ton_iot_v3_ensemble_attack_types.csv',index=False)
service_result.to_csv(METRICS/'ton_iot_v3_ensemble_service_missingness.csv',index=False)
protocol_result.to_csv(METRICS/'ton_iot_v3_ensemble_protocol_errors.csv',index=False)
resources.to_csv(METRICS/'ton_iot_v3_ensemble_resources.csv',index=False)

fig,ax=plt.subplots(figsize=(9,5.5))
for row in comparison.itertuples(index=False):
    ax.scatter(row.false_positive_pct,row.attack_f1_pct,s=65)
    ax.annotate(row.candidate,(row.false_positive_pct,row.attack_f1_pct),
                xytext=(4,5),textcoords='offset points',fontsize=8)
ax.set_xlabel('False-positive rate (%)')
ax.set_ylabel('Attack F1-score (%)')
ax.set_title('ToN-IoT — Single models vs ensembles (validation)')
ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig(FIGURES/'ton_iot_v3_ensemble_tradeoff.png',dpi=170,bbox_inches='tight')
plt.show();plt.close(fig)

missed_lgb=attack & (lgb==0)
rescued=[int((missed_lgb & (predictions[other]==1)).sum()) for other in ('decision_tree','random_forest')]
fig,ax=plt.subplots(figsize=(6,4))
bars=ax.bar(['Decision Tree','Random Forest'],rescued)
for bar,count in zip(bars,rescued):
    ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.2,str(count),ha='center',va='bottom')
ax.set_ylim(0,max(rescued+[1])*1.25)
ax.set_ylabel('LightGBM missed attacks detected by another model')
ax.set_title('ToN-IoT — Different model errors')
fig.tight_layout()
fig.savefig(FIGURES/'ton_iot_v3_ensemble_error_rescues.png',dpi=170,bbox_inches='tight')
plt.show();plt.close(fig)

summary={
    'dataset':'ToN-IoT','feature_set':'Core10','features':FEATURES,
    'train_rows':EXPECTED['train'],'validation_rows':EXPECTED['validation'],
    'test_scored':False,'frozen_manifest_sha256':split['manifest_sha256'],
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'shortlisted_models':list(SHORTLIST),
    'ensemble_count':len(variants),'resource_benchmark_single_flow_runs':BENCH_RUNS,
    'comparison':comparison.to_dict('records'),
    'diversity':diversity.to_dict('records'),
    'service_train_validation_shift':shift.to_dict('records'),
    'resources':resources.to_dict('records'),
    'notes':[
        'Validation-only ensemble screening; no held-out test access.',
        'MITM and other per-type recall are record-level results, not APT campaign-level detection.',
        'Normal and attack labels are compared on the same frozen validation row IDs.',
        'Missing service might encode collection/dataset shortcuts; needs ablation testing.',
        'Group-disjoint feature fingerprints do not guarantee independent capture sessions.',
        'Resource benchmarks are laptop saved-pipeline measurements; RAM includes loaded data and libraries.',
        'Model labels and feature meanings must be harmonized before cross-dataset testing.'
    ]
}
(AUDITS/'ton_iot_v3_ensemble_screening.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print('Saved 7 CSV tables, 2 graphs, 1 JSON report. No retraining, no test access.')